## Day 2 — Cross-Validation 

In [20]:
from sklearn.model_selection import cross_val_score
import pandas as pd

In [21]:
df = pd.read_csv('Housing Prices/housing.csv')

In [22]:
from sklearn.preprocessing import LabelEncoder
lb = LabelEncoder()
df['ocean_proximity'] = lb.fit_transform(df['ocean_proximity'])

In [23]:
df["total_bedrooms"] = df["total_bedrooms"].fillna(df["total_bedrooms"].median())

In [24]:
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]

In [25]:
from sklearn.model_selection import train_test_split
# Step 1: split off 60% train, 40% temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)

In [26]:
# Step 2: split the 40% temp into 20% val, 20% test (i.e. 50/50 of the temp set)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [28]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42)

# 5-fold cross-validation using negative RMSE (sklearn convention)
cv_scores = cross_val_score(
    model, X_train_scaled, y_train,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

rmse_scores = -cv_scores  # convert back to positive values

print("RMSE per fold:", rmse_scores)
print(f"\nMean RMSE: {rmse_scores.mean():.2f}")
print(f"Standard Deviation: {rmse_scores.std():.2f}")

RMSE per fold: [50013.90297655 53111.04011843 47589.86062014 52448.82358788
 51921.87420413]

Mean RMSE: 51017.10
Standard Deviation: 2000.27


Instead of trusting a single train/test split, 5-Fold Cross-Validation 
trains and tests the model five times, rotating which slice of data 
serves as the test set each time — giving a more reliable estimate of 
performance.

`cross_val_score` internally maximizes scores, so RMSE (lower is better) 
is returned as negative (`neg_root_mean_squared_error`) and flipped back 
with `rmse_scores = -cv_scores`.

A low standard deviation across folds means performance is stable, not 
dependent on one lucky split.

**Mean RMSE: 51,017.10**
**Standard Deviation: 2,000.27**

The standard deviation is small relative to the mean (~4%), indicating 
consistent performance across folds rather than a result dependent on 
one particular data split.

| Evaluation Method | RMSE |
|---|---|
| Single train/test split | 51,339.26 |
| 5-Fold Cross-Validation (Mean) | 51,017.10 |

The two estimates are very close (a difference of ~322, under 1%), and 
both fall well within the cross-validation standard deviation (±2,000.27). 
This indicates the single-split score was not a lucky or unlucky outlier 
— it's a reliable estimate that closely matches the more robust 
cross-validated result.